## Prototype

---

> **In one line.** A clone function $f$ copies an existing object $x$ into a brand-new, fully independent object $x'$ — same data ($\text{val}(x') = \text{val}(x)$) but a different memory address ($\text{addr}(x') \neq \text{addr}(x)$), so mutating one never touches the other.

### 1. Objects in memory

Let $\mathcal{O}$ be the set of all objects that exist in memory at runtime. Every object $x \in \mathcal{O}$ carries two distinct properties we must keep apart. Its **value** $\text{val}(x)$ is the totality of its attribute data — everything it *says* (e.g. colour $=$ "red", model $=$ "Toyota"). Its **address** $\text{addr}(x)$ is where it physically *lives* in RAM. Two objects can agree completely on value yet sit at different addresses; this gap between *what an object is* and *where it is* is the entire subject of the Prototype pattern.

The original object — the **prototype** being copied — is $x \in \mathcal{O}$. The copy we produce is the **clone** $x' \in \mathcal{O}$, a new and independent object. An object is rarely flat: write $x_i$ for a nested object inside $x$ (a list, or a sub-object it contains) and $x'_i$ for the corresponding nested object inside the clone. Finally let $m$ denote any **mutation** applied to an object, e.g. setting `x'.colour = "blue"`.

### 2. The clone function

Cloning is a map from objects to objects, $f : \mathcal{O} \rightarrow \mathcal{O}$, sending a prototype to its copy:

$$\boxed{\,f(x) = x', \qquad \text{val}(x') = \text{val}(x), \qquad \text{addr}(x') \neq \text{addr}(x)\,}$$

The clone must satisfy three facts simultaneously: it is a genuinely new object ($\text{addr}(x') \neq \text{addr}(x)$), it begins life with exactly the prototype's data ($\text{val}(x') = \text{val}(x)$), and — the property that makes the copy *useful* — it is **independent**. Independence is stated as a guarantee about mutation: applying any change to the clone leaves the original's value alone,

$$m(x') \;\Longrightarrow\; \text{val}(x) \text{ unchanged} \qquad \text{(independence condition)}.$$

The flow is a single fork: one prototype enters, a separate object exits, and a later edit lands on only one branch.

$$\underbrace{x}_{\text{prototype}} \;\xrightarrow{\;f\;}\; \underbrace{x'}_{\text{clone},\ \text{val}=\text{val}(x)} \;\xrightarrow{\;m\;}\; \underbrace{m(x')}_{\text{val}(x)\ \text{still intact}}$$

### 3. Key conditions

1. **Value equality** — the clone starts with the same data as the original:
   $$\text{val}(x') = \text{val}(x).$$
2. **Address inequality** — they are two separate objects in RAM, not the same one:
   $$\text{addr}(x') \neq \text{addr}(x).$$
3. **Independence (deep copy)** — independence must hold *recursively* down through every nested object, not just at the top level:
   $$\text{addr}(x'_i) \neq \text{addr}(x_i) \quad \text{for all } i.$$
   A shallow copy violates this: it duplicates the outer shell but lets $x'_i$ and $x_i$ share one address, so editing the clone's list leaks back into the original.
4. **Chained cloning** — a clone is itself a valid prototype, so $f$ composes and every link stays independent:
   $$f(x) = x', \quad f(x') = x'', \qquad \text{addr}(x) \neq \text{addr}(x') \neq \text{addr}(x'').$$

&nbsp;

> 🧬 Think of biological cell division. A cell $x$ splits into $x'$ — same DNA ($\text{val}(x') = \text{val}(x)$), but a completely separate physical cell ($\text{addr}(x') \neq \text{addr}(x)$). Damage to $x'$ does not damage $x$. They share an origin but are now fully independent.

### Exercise 09 — Basic Clone

---

**Scenario:** A `Car` object took a lot of setup. You want a near-identical second car. Cloning avoids rebuilding from scratch while keeping the two objects fully independent.

**Your task:** Create `Car` with a `clone()` method satisfying $f(x) = x'$ where $\text{val}(x') = \text{val}(x)$ and $\text{addr}(x') \neq \text{addr}(x)$.

```python
car1 = Car("Toyota", "red", "V6")   # x
car2 = car1.clone()                 # x' = f(x)
car2.color = "blue"                 # m(x')
print(car1.color)                   # red  -- val(x) unchanged
print(car2.color)                   # blue -- x' is independent
```

**Hints**

- `clone()` returns `copy.deepcopy(self)` — deep copy satisfies the independence condition for all nested $x_i$.
- Test with a nested list attribute to confirm deep copy is needed: modify the list in $x'$ and verify $x$'s list is untouched.

In [ ]:
import copy

# --------------------------------
# Prototype: f(x) = x' with val(x') = val(x) and addr(x') != addr(x)

class Car:
    def __init__(self, model, color, engine):
        self.model = model
        self.color = color
        self.engine = engine
        self.features = []           # nested object x_i -- needs deep copy

    def clone(self):                 # f(x) -> x'
        # return an independent copy: same val(...), different addr(...)
        # deep copy ensures addr(x'_i) != addr(x_i) for nested features too
        ...

# --------------------------------
car1 = Car("Toyota", "red", "V6")    # x
car2 = car1.clone()                  # x' = f(x)
car2.color = "blue"                  # m(x')
print(car1.color)                    # red  -- val(x) unchanged
print(car2.color)                    # blue -- x' is independent

### Exercise 10 — Prototype Registry

---

**Scenario:** A catalog stores pre-configured prototype objects. Users request a clone by name, customise it, and use it — the original in the registry is never touched.

**Your task:** Build a `ShapeRegistry` that stores prototypes $x$ and returns clones $x' = f(x)$ on request.

```python
registry = ShapeRegistry()
registry.register("circle", Circle(radius=5, color="red"))   # store x
my_circle = registry.get_clone("circle")                     # x' = f(x)
my_circle.color = "green"                                    # m(x')
print(registry.get_clone("circle").color)                    # red -- x unchanged
```

**Hints**

- `get_clone(name)` returns `copy.deepcopy(self._prototypes[name])` — never the original $x$ directly, always $f(x) = x'$.
- The registry dictionary maps $\text{name} \rightarrow x$. Think of it as a library of blueprints — you always photocopy the blueprint, never hand out the original.

In [ ]:
import copy

# --------------------------------
# A concrete prototype object -- you do not change this

class Circle:
    def __init__(self, radius, color):
        self.radius = radius
        self.color = color

# --------------------------------
# Registry of prototypes: name -> x, hands out x' = f(x) on request

class ShapeRegistry:
    def __init__(self):
        self._prototypes = {}        # maps name -> x (the stored prototype)

    def register(self, name, prototype):
        # store x under its name
        ...

    def get_clone(self, name):       # returns x' = f(x), never x itself
        # deep copy the stored prototype so addr(x') != addr(x)
        ...

# --------------------------------
registry = ShapeRegistry()
registry.register("circle", Circle(radius=5, color="red"))   # store x
my_circle = registry.get_clone("circle")                     # x' = f(x)
my_circle.color = "green"                                    # m(x')
print(registry.get_clone("circle").color)                    # red -- x unchanged